# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/notebooks/02_your_first_readable_model.ipynb)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [ ]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

# ── Hand-built rule ──
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]
top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
print("\nTop 10 by hand rule:")
print(top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
print("\n=== Hand rule precision ===")
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

# ── Baseline tree (max_depth=2) ──
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)
print("\n=== Baseline tree (max_depth=2) ===")
print(export_text(tree, feature_names=features))

tree_score = tree.predict_proba(X)[:, 1]
print("\n=== Hand rule vs. tree, in-sample ===")
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

# ── Leaky tree (uses trend_pct, which is derived from the label — for illustration only) ──
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"\n'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))


# =========================================================
# QUESTION 1: Does increasing max_depth improve Precision@50?
#             Can you still read the tree?
# =========================================================
print("\n\n=== Q1: Varying max_depth ===")
for depth in (2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    t.fit(X, y)
    score = t.predict_proba(X)[:, 1]
    p20 = precision_at_k(score, y, 20)
    p50 = precision_at_k(score, y, 50)
    print(f"max_depth={depth}  leaves={t.get_n_leaves():2d}  "
          f"Precision@20={p20:.3f}  Precision@50={p50:.3f}")

print("\n--- depth=4 tree printed out (readability check) ---")
tree4 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X, y)
print(export_text(tree4, feature_names=features))
print("Leaves at depth=4:", tree4.get_n_leaves(),
      "-> compare this to depth=2's leaf count above; more leaves means harder to summarize in a sentence.")


# =========================================================
# QUESTION 2: Swap features — drop impressions_90d, add engagement_rate.
#             What does the tree split on first?
# =========================================================
print("\n\n=== Q2: Feature swap (-impressions_90d, +engagement_rate) ===")
features_v2 = ["content_age_days", "days_since_last_update", "avg_position",
               "ctr", "word_count", "engagement_rate"]

X2 = df[features_v2].replace([np.inf, -np.inf], np.nan).fillna(0)
tree_v2 = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_v2.fit(X2, y)

print(export_text(tree_v2, feature_names=features_v2))

score_v2 = tree_v2.predict_proba(X2)[:, 1]
for k in (20, 50):
    print(f"Precision@{k} with engagement_rate swap: {precision_at_k(score_v2, y, k):.3f}")

root_feature_idx = tree_v2.tree_.feature[0]
print(f"\nRoot split feature: {features_v2[root_feature_idx]}  "
      f"(threshold={tree_v2.tree_.threshold[0]:.2f})")


# =========================================================
# QUESTION 3: Client-holdout split — does the in-sample gap hold?
# =========================================================
print("\n\n=== Q3: Client-holdout validation ===")

# Confirm the client identifier column name before running — adjust if needed
candidate_cols = [c for c in df.columns if "client" in c.lower() or "site" in c.lower() or "domain" in c.lower()]
print("Candidate client-identifier columns found:", candidate_cols)

client_col = "client_id"  # <-- update this if the printed list above shows a different name

if client_col in df.columns:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups=df[client_col]))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    df_test = df.iloc[test_idx]

    tree_holdout = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
    tree_holdout.fit(X_train, y_train)

    test_score = tree_holdout.predict_proba(X_test)[:, 1]
    hand_rule_test = df_test["hand_rule_score"].values

    print("\nIn-sample (original, for comparison):")
    for k in (20, 50):
        hr_in = precision_at_k(df["hand_rule_score"], y, k)
        tr_in = precision_at_k(tree_score if False else tree.predict_proba(X)[:, 1], y, k)
        print(f"  Precision@{k}:  hand rule {hr_in:.3f}   vs   tree {tr_in:.3f}")

    print("\nHeld-out (client never in both train & test):")
    for k in (20, 50):
        hr_out = precision_at_k(hand_rule_test, y_test, k)
        tr_out = precision_at_k(test_score, y_test, k)
        print(f"  Precision@{k}:  hand rule {hr_out:.3f}   vs   tree {tr_out:.3f}")
else:
    print(f"\nColumn '{client_col}' not found in df. Check scripts/03_train_model.py "
          f"for the correct client-grouping column and update `client_col` above.")


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.